# Eval: Model Selector Agent (Contribution 2)

Compares one-shot baseline vs decision tree on 20 natural language queries.
Requires API key. Run after setting up .env.yaml.

In [ ]:
import os, json
from functions.generic_tools import set_env_vars
set_env_vars('./.env.yaml')
from openai import AzureOpenAI
from utils.model_selector import LOOKUP, select_model

client = AzureOpenAI(
    api_key=os.environ['OPENAI_API_KEY'],
    azure_endpoint=os.environ['AZURE_API_URL'],
    api_version=os.environ['AZURE_API_VERSION'],
)
MODEL = os.environ['MODEL_NAME']
zoo = json.load(open('model_zoo.json'))
ALL_MODELS = [m['id'] for cat in zoo.values() if isinstance(cat, list) for m in cat if isinstance(m, dict)]
print(f'Total models in zoo: {len(ALL_MODELS)}')

In [ ]:
# 20 natural language queries a real user would type
# Includes synonyms, vague phrasing, domain language — not just clean keywords
EVAL_TASKS = [
    # Clear queries
    ('Classify rice leaf images for nitrogen deficiency severity',
     'Naren1704/rice_nutrient-deficiency_rgbdataset_dinov2_fullft'),
    ('Detect and count panicles in UAV rice field images',
     'Naren1704/rice_panicle-detection_bangladesh_yolov8m_fullft'),
    ('Segment arabidopsis leaves and count them',
     'fengchen025/arabidopsis_leaf-instance-segmentation_cvppp2017-a1a4_m2fb_fullft'),
    ('Classify wheat canopy drone images for nutrient stress',
     'Naren1704/wheat_nutrient-deficiency_rgbdataset_dinov2_fullft'),
    ('Analyse Sentinel-2 time series to predict crop type and harvest timing',
     'Naren1704/sentinel2_crop-mapping_lombardia_bilstm_multitask'),
    ('Classify banana leaves for micronutrient deficiency',
     'Naren1704/banana_nutrient-deficiency_rgbdataset_dinov2_fullft'),
    ('Count wheat heads in field images',
     'Naren1704/wheat_spike-detection_gwhd_yolov8m_fullft'),
    ('Identify nutrient deficiency in coffee plant leaves',
     'Naren1704/coffee_nutrient-deficiency_rgbdataset_dinov2_fullft'),
    ('Classify maize corn leaves for nutrient stress',
     'Naren1704/maize_nutrient-deficiency_rgbdataset_dinov2_fullft'),
    ('Segment potato leaves from close-range RGB images',
     'potato_leaf-instance-segmentation_leaf-only-sam'),
    # Synonym / domain language queries
    ('My paddy crop leaves look yellow, identify the deficiency',
     'Naren1704/rice_nutrient-deficiency_rgbdataset_dinov2_fullft'),
    ('Count the number of ears in these wheat photos taken from the ground',
     'Naren1704/wheat_spike-detection_gwhd_yolov8m_fullft'),
    ('I have drone images of my wheat field, classify nutrient issues',
     'Naren1704/wheat_nutrient-deficiency_rgbdataset_dinov2_fullft'),
    ('Run crop type mapping on satellite imagery time series',
     'Naren1704/sentinel2_crop-mapping_lombardia_bilstm_multitask'),
    ('Detect rosette leaves in arabidopsis images',
     'fengchen025/arabidopsis_leaf-instance-segmentation_cvppp2017-a1a4_m2fb_fullft'),
    # Vague / partial queries
    ('Analyse these rice UAV images for reproductive organ counting',
     'Naren1704/rice_panicle-detection_bangladesh_yolov8m_fullft'),
    ('My banana plants show yellowing and leaf curl, what deficiency?',
     'Naren1704/banana_nutrient-deficiency_rgbdataset_dinov2_fullft'),
    ('Phenotype potato leaves',
     'potato_leaf-instance-segmentation_leaf-only-sam'),
    ('Check maize images for any nutritional problems',
     'Naren1704/maize_nutrient-deficiency_rgbdataset_dinov2_fullft'),
    ('Predict growth rate and harvest time from multi-temporal field data',
     'Naren1704/sentinel2_crop-mapping_lombardia_bilstm_multitask'),
]
print(f'Evaluation tasks: {len(EVAL_TASKS)}')

In [ ]:
# BASELINE: one-shot selection (original behaviour)
# Manager sees all model IDs at once and picks one
def baseline_select(query):
    model_list = '\n'.join(f'- {m}' for m in ALL_MODELS)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{
            'role': 'user',
            'content': f'Task: "{query}"\n\nAvailable models:\n{model_list}\n\nReply with ONLY the single model ID that best fits this task. Nothing else.'
        }],
        temperature=0, max_tokens=80,
    )
    return resp.choices[0].message.content.strip()

print('Running baseline (one-shot)...')
baseline_results = []
for query, expected in EVAL_TASKS:
    predicted = baseline_select(query)
    correct = expected in predicted
    baseline_results.append(correct)
    status = 'PASS' if correct else 'FAIL'
    print(f'[{status}] {query[:55]}...')
    if not correct:
        print(f'       Expected: {expected.split("/")[-1]}')
        print(f'       Got:      {predicted[:80]}')

baseline_acc = sum(baseline_results) / len(baseline_results)
print(f'\nBaseline accuracy: {baseline_acc:.0%} ({sum(baseline_results)}/{len(baseline_results)})')

In [ ]:
# CONTRIBUTION 2: decision tree selection
print('Running Contribution 2 (decision tree)...')
tree_results = []
for query, expected in EVAL_TASKS:
    predicted = select_model(query, client, MODEL)
    correct = predicted == expected
    tree_results.append(correct)
    status = 'PASS' if correct else 'FAIL'
    print(f'[{status}] {query[:55]}...')
    if not correct:
        print(f'       Expected: {expected.split("/")[-1]}')
        print(f'       Got:      {str(predicted)[:80]}')

tree_acc = sum(tree_results) / len(tree_results)
print(f'\nDecision tree accuracy: {tree_acc:.0%} ({sum(tree_results)}/{len(tree_results)})')

In [ ]:
# Summary table
import pandas as pd
rows = []
for i, (query, expected) in enumerate(EVAL_TASKS):
    rows.append({
        'query': query,
        'expected': expected.split('/')[-1],
        'baseline_correct': baseline_results[i],
        'tree_correct': tree_results[i],
    })
df = pd.DataFrame(rows)
print(df[['query','baseline_correct','tree_correct']].to_string(index=False))
print(f'\nBaseline:        {baseline_acc:.0%}')
print(f'Decision tree:   {tree_acc:.0%}')
print(f'Improvement:     +{(tree_acc - baseline_acc):.0%}')
df.to_csv('results/eval_model_selector.csv', index=False)
print('Saved to results/eval_model_selector.csv')